# Safebooru 메타데이터 크롤링
Safebooru API에서 메타데이터(태그 + URL)만 수집해 Parquet으로 저장합니다.
- 1000건마다 `1000.parquet`, `2000.parquet`, ... 형태로 순차 저장
- 중단 후 이어서 크롤링 가능 (기존 파일 자동 감지)
- **이미지는 저장하지 않음** — 학습 시 배치 단위로 임시 다운로드

In [ ]:
import pandas as pd
import os
import time
import requests
import urllib3
import ssl
import re
from requests.adapters import HTTPAdapter
from tqdm.auto import tqdm

# SSL 경고 무시
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class TLSAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.check_hostname = False
        ctx.verify_mode = ssl.CERT_NONE
        ctx.set_ciphers('DEFAULT@SECLEVEL=1')
        kwargs['ssl_context'] = ctx
        return super(TLSAdapter, self).init_poolmanager(*args, **kwargs)

# --- [경로 설정] ---
SAVE_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "parquet"))
os.makedirs(SAVE_DIR, exist_ok=True)

# 인증 정보
USER_ID = "925305"
API_KEY = "b462a643571c0dd74c57131f82078b100dec94bd0bfd118fb0e7eb9b95b7182a"
BASE_URL = "https://gelbooru.com/index.php"

def get_last_state():
    """저장된 파일들을 분석하여 마지막 파일 번호와 가장 작은 ID(기준점)를 반환"""
    files = [f for f in os.listdir(SAVE_DIR) if f.endswith('.parquet')]
    if not files:
        return None, 0
    
    # 파일명에서 숫자 추출 후 정렬
    file_nums = sorted([int(re.search(r'(\d+)', f).group(1)) for f in files])
    last_file_num = file_nums[-1]
    
    # 마지막 저장 파일에서 가장 작은 ID를 찾아야 그 다음(과거) 데이터를 가져옴
    try:
        df = pd.read_parquet(os.path.join(SAVE_DIR, f"{last_file_num}.parquet"))
        min_id = df['id'].min()
        return int(min_id), last_file_num
    except:
        return None, 0

def crawl_gelbooru_all_tags():
    session = requests.Session()
    session.mount('https://', TLSAdapter())
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
    })

    # 1. 마지막 작업 상태 불러오기
    last_id, current_file_count = get_last_state()
    
    # 2. 전체 포스트 수 파악 (진행률 표시용)
    try:
        init_res = session.get(BASE_URL, params={'page': 'dapi', 's': 'post', 'q': 'index', 'json': 1, 'limit': 1}, verify=False)
        total_posts = int(init_res.json().get('@attributes', {}).get('count', 0))
    except:
        total_posts = 1000000

    print(f"전체 대상 데이터: 약 {total_posts}개")
    print(f"수집 시작 ID 기준점: {last_id if last_id else '최신 데이터부터'}")

    pbar = tqdm(total=total_posts // 100, desc="Gelbooru Full Tag Scraping")
    
    # 이미 받은 파일이 있다면 pbar 위치 조정
    if current_file_count > 0:
        pbar.update(current_file_count // 100)

    while True:
        current_file_count += 100
        file_path = os.path.join(SAVE_DIR, f"{current_file_count}.parquet")
        
        # 핵심: id:<{last_id} 필터를 사용하여 Deep Paging 에러 우회
        tag_query = f"id:<{last_id}" if last_id else ""
        
        params = {
            'page': 'dapi', 's': 'post', 'q': 'index', 'json': 1,
            'limit': 100, 'pid': 0, 'tags': tag_query,
            'api_key': API_KEY, 'user_id': USER_ID
        }
        
        try:
            response = session.get(BASE_URL, params=params, timeout=30, verify=False)
            
            if response.status_code == 200:
                # 'Too deep' 메시지 포함 여부 확인
                if "Too deep" in response.text:
                    print(f"\n[오류] ID 기반 조회임에도 Deep Paging 에러 발생. 쿼리를 확인하세요.")
                    break
                
                if not response.text.strip():
                    print(f"\n[{current_file_count}] 빈 응답. 잠시 대기...")
                    time.sleep(10)
                    continue

                res_data = response.json()
                posts = res_data.get('post', []) if isinstance(res_data, dict) else []
                
                if posts:
                    df = pd.DataFrame(posts)
                    # ID를 정수형으로 변환하여 최소값 갱신 준비
                    df['id'] = df['id'].astype(int)
                    last_id = df['id'].min()
                    
                    # 모든 태그를 포함하여 필요한 컬럼만 추출
                    # 'tags' 컬럼에 해당 이미지의 모든 태그가 문자열로 들어있음
                    target_cols = ['id', 'tags', 'file_url', 'sample_url', 'width', 'height', 'rating', 'source']
                    actual_cols = [c for c in target_cols if c in df.columns]
                    
                    df[actual_cols].to_parquet(file_path, engine='pyarrow', index=False)
                    
                    pbar.set_postfix(min_id=last_id)
                    pbar.update(1)
                else:
                    print(f"\n[{current_file_count}] 더 이상 가져올 데이터가 없습니다. 수집 완료.")
                    break
            
            elif response.status_code == 429:
                print(f"\n[429] 과도한 요청. 60초간 정지합니다.")
                time.sleep(60)
                continue
            else:
                print(f"\n[Error] {response.status_code} 응답. 대기 후 재시도.")
                time.sleep(10)
                continue
                
        except Exception as e:
            print(f"\n[Exception] {e}")
            time.sleep(10)
            continue
            
        # 서버 부하 방지 및 차단 회피를 위한 대기 시간 (3초 권장)
        time.sleep(3.0)

    pbar.close()

if __name__ == "__main__":
    crawl_gelbooru_all_tags()

서버 전체 데이터: Unknown
수집 재개 ID: 13838923


Gelbooru Crawling (ID-based): 0it [00:00, ?it/s]

KeyboardInterrupt: 